In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Pérdida
El algoritmo de dice-loss es ideal para este tipo de problema porque evalua el error en base al empalme de la predicción con el valor esperado. Como pudimos ver en el análisis de la fáse "Exploratory Data Analysis", algunas veces la segmentación de imágenes pueden ser fragmentos muy pequeños de la imagen. 

Por ello, un algoritmo capaz de identificar el error en base a cuantos pixeles estamos bien contra la cantidad de pixeles con la que estamos mal es una buena estrategia para poder mejorar el modelo y encontrar la mejor segmentación en la imagen. 

In [3]:
class DiceLoss(nn.Module):
  def __init__(self, num_classes = 3, smooth= 1.0):
    super().__init__()
    self.num_classes = num_classes
    self.smooth = smooth

  def forward(self, predictions, targets):
    predictions = torch.softmax(predictions, dim=1)

    tagret_onehot = F.one_hot(targets, num_classes=self.num_classes)
    tagret_onehot = tagret_onehot.permute(0, 3, 1, 2).float()

    dims = (0,2,3)
    intersection = torch.sum(predictions * tagret_onehot, dims)
    union = predictions.sum(dims) + tagret_onehot.sum(dims)

    dice_per_class = (2. * intersection + self.smooth) / (union + self.smooth)
    return 1 - dice_per_class.mean()